In [27]:
# %pip install langchain==0.3.20 | tail -n 1 
# %pip install crewai==0.80.0 | tail -n 1
# %pip install langchain-community==0.3.19 | tail -n 1 
# %pip install crewai-tools==0.38.0 | tail -n 1
# %pip install databricks-sdk==0.57.0| tail -n 1

In [28]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_ORG_ID = os.getenv("OPENAI_ORG_ID", "")

# Model Configuration
DEFAULT_MODEL = os.getenv("DEFAULT_MODEL", "gpt-3.5-turbo")
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.7"))
MAX_TOKENS = int(os.getenv("MAX_TOKENS", "500"))
WATSON_URL = os.getenv("IBM_URL_END_POINT")
PROJECT_ID = os.getenv("IBM_PROJECT_ID")
IBM_API_KEY = os.getenv("IBM_API_KEY")
TAVILY_KEY = os.getenv("TAVILY_API_KEY")
SERPER_API_KEY = os.getenv("SERPER_API_KEY")

In [29]:
import os 
os.environ['SERPER_API_KEY'] = SERPER_API_KEY

In [30]:
%%capture

from crewai_tools import SerperDevTool

In [31]:
search_tool=SerperDevTool()
print(type(search_tool))

<class 'crewai_tools.tools.serper_dev_tool.serper_dev_tool.SerperDevTool'>


In [32]:
search_query = "Latest Breakthroughs in machine learning"
search_results =search_tool.run(query=search_query )

# Print the results
print(f"Search Results for '{search_results}':\n")

Using Tool: Search the internet with Serper
Search Results for '{'searchParameters': {'q': 'Latest Breakthroughs in machine learning', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Advancements in AI and Machine Learning', 'link': 'https://ep.jhu.edu/news/advancements-in-ai-and-machine-learning/', 'snippet': 'AI and ML advancements are transforming engineering by automating complex tasks and enhancing decision-making processes for professionals.', 'position': 1}, {'title': 'Latest AI News and AI Breakthroughs that Matter Most: 2026', 'link': 'https://www.crescendo.ai/news/latest-ai-news-and-updates', 'snippet': "Wondering what's happening in the AI world? Here are the latest AI breakthroughs and news that are shaping the world around us!", 'position': 2}, {'title': '5 Breakthrough Machine Learning Research Papers Already in 2025', 'link': 'https://machinelearningmastery.com/5-breakthrough-machine-learning-research-papers-already-in-2025/', 'snippet': '1. SAM 

In [33]:
print("keys of search_results", search_results.keys())

keys of search_results dict_keys(['searchParameters', 'organic', 'relatedSearches', 'credits'])


In [34]:
from crewai import LLM

llm = LLM(
    model="openai/gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    max_tokens=2000
)

In [35]:
llm

In [36]:
from crewai import Agent

research_agent = Agent(
  role='Senior Research Analyst',
  goal='Uncover cutting-edge information and insights on any subject with comprehensive analysis',
  backstory="""You are an expert researcher with extensive experience in gathering, analyzing, and synthesizing information across multiple domains. 
  Your analytical skills allow you to quickly identify key trends, separate fact from opinion, and produce insightful reports on any topic. 
  You excel at finding reliable sources and extracting valuable information efficiently.""",
  verbose=True,
  allow_delegation=False,
  llm = llm,
  tools=[SerperDevTool()]
)

In [37]:
research_agent

Agent(role=Senior Research Analyst, goal=Uncover cutting-edge information and insights on any subject with comprehensive analysis, backstory=You are an expert researcher with extensive experience in gathering, analyzing, and synthesizing information across multiple domains. 
  Your analytical skills allow you to quickly identify key trends, separate fact from opinion, and produce insightful reports on any topic. 
  You excel at finding reliable sources and extracting valuable information efficiently.)

In [38]:
# Define your agents with roles and goals
# Define the Writer Agent
writer_agent = Agent(
  role='Tech Content Strategist',
  goal='Craft well-structured and engaging content based on research findings',
  backstory="""You are a skilled content strategist known for translating 
  complex topics into clear and compelling narratives. Your writing makes 
  information accessible and engaging for a wide audience.""",
  verbose=True,
  llm = llm,
  allow_delegation=True
)

In [39]:
writer_agent

Agent(role=Tech Content Strategist, goal=Craft well-structured and engaging content based on research findings, backstory=You are a skilled content strategist known for translating 
  complex topics into clear and compelling narratives. Your writing makes 
  information accessible and engaging for a wide audience.)

In [40]:
from crewai import Task

research_task = Task(
  description="Analyze the major {topic}, identifying key trends and technologies. Provide a detailed report on their potential impact.",
  agent=research_agent,
  expected_output="A detailed report on {topic}, including trends, emerging technologies, and their impact."
)

In [41]:
# Create a task for the Writer Agent
writer_task = Task(
  description="Create an engaging blog post based on the research findings about {topic}. Tailor the content for a tech-savvy audience, ensuring clarity and interest.",
  agent=writer_agent,
  expected_output="A 4-paragraph blog post on {topic}, written clearly and engagingly for tech enthusiasts."
)

In [42]:
from crewai import Crew, Process

crew = Crew(
    agents=[research_agent, writer_agent],
    tasks=[research_task, writer_task],
    process=Process.sequential,
    verbose=True 
)

In [43]:
result = crew.kickoff(inputs={"topic": "Latest Generative AI breakthroughs"})

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e616a047-c05e-44c2-a689-f7bc2bd375f0                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Task: Analyze the major Latest Generative AI breakthroughs, identifying key trends and technologies. Provide   │
│  a detailed report on their potential impact.                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Thought: I need to gather the latest information on breakthroughs in generative AI, trends, and emerging       │
│  technologies to provide a comprehensive report on their impact.                                                │
│                                                                                                                 │
│  Using Tool: Search the internet with Serper                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"search_query\": \"latest generative AI breakthroughs 2023 trends technologies impact\"}"                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tech Content Strategist                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Latest Breakthroughs in Generative AI: 2023 Insights**                                                       │
│                                                                                                                 │
│  **1. Overview of Generative AI’s Growth**                                                                      │
│  In 2023, generative AI has established itself as a transformative force across various sectors, marking an     │
│  exponential growth in applications and user adoption. According to McKinsey's Global Survey, generative AI     │
│  tools have gained popularity, being integrated into a myriad of business operations. Over 50% of               │
│  organizations report using generative AI in at least one of their business processes, reflecting its rapid     │
│  acceptance and utility.                                                                                        │
│                                                                                                                 │
│  **2. Notable Technologies**                                                                                    │
│  Several key technologies have emerged this year, enhancing the capabilities of generative AI:                  │
│  - **Transformer Architectures**: Continued advancements in transformer models, which are critical for          │
│  processing and generating human-like text and images. Notably, architecture variations like GPT-4 have made    │
│  headlines for their superior performance in generating contextual responses.                                   │
│  - **Diffusion Models**: Technologies such as DALL-E 2 and Midjourney employ diffusion processes to create      │
│  high-resolution images from textual descriptions, showcasing the breadth of creative applications in media     │
│  and entertainment.                                                                                             │
│  - **Federated Learning & Differential Privacy**: These methods are increasingly employed to enhance user       │
│  privacy while allowing AI systems to learn from decentralized data without breaching confidentiality, which    │
│  is crucial in today’s data-sensitive environment.                                                              │
│                                                                                                                 │
│  **3. Key Statistics**                                                                                          │
│  - The generative AI market is projected to grow at a CAGR of 34.6% from 2023 to 2030, with a forecast of       │
│  generating over $1.3 trillion in economic value annually by 2030 (McKinsey Global Institute).                  │
│  - 30% of organizations plan to integrate generative AI to automate up to 30% of their operational tasks,       │
│  significantly improving workflow efficiency (Simplilearn).                                                     │
│                                                                                                                 │
│  **4. Case Studies**                                                                                            │
│  - **OpenAI’s ChatGPT**: This platform became widely used across businesses for customer service, content       │
│  creation, and software development, indicating its ver

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 2c12a679-821d-4a89-99da-0b40760deaa7                                                                     │
│  Agent: Tech Content Strategist                                                                                 │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e616a047-c05e-44c2-a689-f7bc2bd375f0                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: **Latest Breakthroughs in Generative AI: 2023 Insights**                                         │
│                                                                                                                 │
│  **1. Overview of Generative AI’s Growth**                                                                      │
│  In 2023, generative AI has established itself as a transformative force across various sectors, marking an     │
│  exponential growth in applications and user adoption. According to McKinsey's Global Survey, generative AI     │
│  tools have gained popularity, being integrated into a myriad of business operations. Over 50% of               │
│  organizations report using generative AI in at least one of their business processes, reflecting its rapid     │
│  acceptance and utility.                                                                                        │
│                                                                                                                 │
│  **2. Notable Technologies**                                                                                    │
│  Several key technologies have emerged this year, enhancing the capabilities of generative AI:                  │
│  - **Transformer Architectures**: Continued advancements in transformer models, which are critical for          │
│  processing and generating human-like text and images. Notably, architecture variations like GPT-4 have made    │
│  headlines for their superior performance in generating contextual responses.                                   │
│  - **Diffusion Models**: Technologies such as DALL-E 2 and Midjourney employ diffusion processes to create      │
│  high-resolution images from textual descriptions, showcasing the breadth of creative applications in media     │
│  and entertainment.                                                                                             │
│  - **Federated Learning & Differential Privacy**: These methods are increasingly employed to enhance user       │
│  privacy while allowing AI systems to learn from decentralized data without breaching confidentiality, which    │
│  is crucial in today’s data-sensitive environment.                                                              │
│                                                                                                                 │
│  **3. Key Statistics**                                                                                          │
│  - The generative AI market is projected to grow at a CAGR of 34.6% from 2023 to 2030, with a forecast of       │
│  generating over $1.3 trillion in economic value annually by 2030 (McKinsey Global Institute).                  │
│  - 30% of organizations plan to integrate generative AI to automate up to 30% of their operational tasks,       │
│  significantly improving workflow efficiency (Simplilearn).                                                     │
│                                                                                                                 │
│  **4. Case Studies**                                                                                            │
│  - **OpenAI’s ChatGPT**: This platform became widely u

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

In [44]:
type(result)

crewai.crews.crew_output.CrewOutput

In [45]:
result

CrewOutput(raw="**Latest Breakthroughs in Generative AI: 2023 Insights**\n\n**1. Overview of Generative AI’s Growth**  \nIn 2023, generative AI has established itself as a transformative force across various sectors, marking an exponential growth in applications and user adoption. According to McKinsey's Global Survey, generative AI tools have gained popularity, being integrated into a myriad of business operations. Over 50% of organizations report using generative AI in at least one of their business processes, reflecting its rapid acceptance and utility.  \n\n**2. Notable Technologies**  \nSeveral key technologies have emerged this year, enhancing the capabilities of generative AI:\n- **Transformer Architectures**: Continued advancements in transformer models, which are critical for processing and generating human-like text and images. Notably, architecture variations like GPT-4 have made headlines for their superior performance in generating contextual responses.\n- **Diffusion Mode

In [46]:
final_output = result.raw
print("Final output:", final_output)

Final output: **Latest Breakthroughs in Generative AI: 2023 Insights**

**1. Overview of Generative AI’s Growth**  
In 2023, generative AI has established itself as a transformative force across various sectors, marking an exponential growth in applications and user adoption. According to McKinsey's Global Survey, generative AI tools have gained popularity, being integrated into a myriad of business operations. Over 50% of organizations report using generative AI in at least one of their business processes, reflecting its rapid acceptance and utility.  

**2. Notable Technologies**  
Several key technologies have emerged this year, enhancing the capabilities of generative AI:
- **Transformer Architectures**: Continued advancements in transformer models, which are critical for processing and generating human-like text and images. Notably, architecture variations like GPT-4 have made headlines for their superior performance in generating contextual responses.
- **Diffusion Models**: Tech

In [48]:
tasks_outputs = result.tasks_output

In [49]:
print("Task Description", tasks_outputs[0].description)
print("Output of research task ",tasks_outputs[0])

Task Description Analyze the major Latest Generative AI breakthroughs, identifying key trends and technologies. Provide a detailed report on their potential impact.
Output of research task  Generative AI continues to transform industries and redefine technological possibilities in 2023. Below, I present a comprehensive analysis of the latest breakthroughs, key trends, and emerging technologies in generative AI, along with their anticipated impact.

1. **Overview of Major Breakthroughs**
   - **Widespread Adoption**: Generative AI has moved from experimental models to being widely adopted across various sectors. According to McKinsey's report, leading organizations are leveraging generative AI to enhance productivity and innovate products (McKinsey, 2023).
   - **Advancements in Models**: Significant improvements in language models and image generators have been recorded. Tools such as OpenAI’s GPT-4 and Google's Bard AI are showcasing enhanced linguistic, contextual understanding, and 

In [50]:
print("Writer task description:", tasks_outputs[1].description)
print(" \nOutput of writer task:", tasks_outputs[1].raw)

Writer task description: Create an engaging blog post based on the research findings about Latest Generative AI breakthroughs. Tailor the content for a tech-savvy audience, ensuring clarity and interest.
 
Output of writer task: **Latest Breakthroughs in Generative AI: 2023 Insights**

**1. Overview of Generative AI’s Growth**  
In 2023, generative AI has established itself as a transformative force across various sectors, marking an exponential growth in applications and user adoption. According to McKinsey's Global Survey, generative AI tools have gained popularity, being integrated into a myriad of business operations. Over 50% of organizations report using generative AI in at least one of their business processes, reflecting its rapid acceptance and utility.  

**2. Notable Technologies**  
Several key technologies have emerged this year, enhancing the capabilities of generative AI:
- **Transformer Architectures**: Continued advancements in transformer models, which are critical fo

In [51]:
print("We can get the agent for researcher task:  ",tasks_outputs[0].agent)
print("We can get the agent for the writer task: ",tasks_outputs[1].agent)

We can get the agent for researcher task:   Senior Research Analyst
We can get the agent for the writer task:  Tech Content Strategist


In [52]:
token_count = result.token_usage.total_tokens
prompt_tokens = result.token_usage.prompt_tokens
completion_tokens = result.token_usage.completion_tokens

print(f"Total tokens used: {token_count}")
print(f"Prompt tokens: {prompt_tokens} (used for instructions to the model)")
print(f"Completion tokens: {completion_tokens} (generated in response)")

Total tokens used: 10808
Prompt tokens: 8178 (used for instructions to the model)
Completion tokens: 2630 (generated in response)


In [55]:
social_agent = Agent(
    role='Social Media Strategist',
    goal='Generate engaging social media snippets based on the full article',
    backstory="A digital storyteller who excels at crafting compelling posts to drive engagement and traffic.",
    verbose=True
)

In [57]:
social_task = Task(
    description=(
        "Summarize the blog post about {topic} into 2–3 engaging social media posts "
        "suitable for platforms like LinkedIn or Twitter. Make sure the tone is informative, "
        "professional, and encourages further reading."
    ),
    agent=social_agent,
    expected_output="A series of 2–3 well-written social posts highlighting the key insights from the blog content."
)

In [ ]:
crew = Crew(
    agents=[research_agent, writer_agent, social_agent],
    tasks=[research_task, writer_task, social_task],
    process=Process.sequential,  # Tasks will be executed one after another
    verbose=True
)

# Run the crew and capture the final output (includes research, blog post, and social media content)
result = crew.kickoff(inputs={"topic": "Latest Generative AI breakthroughs"})